In [1]:
import torch
import re
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn

/Users/yashchaudhary/Documents/nlp-document-classifier/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
PAD_ID = 0
UNK_ID = 1
CHUNK_SIZE = 3
embedding_dim = 4

sentence = "Hello, I am learning NLP techniques to classify documents{}."

In [3]:
def clean_text(txt):

    text = txt.lower().replace(".", "")

    text = re.sub(r"[^\w\s]", "", text)

    return " ".join(text.split())


def tokenize_encode(txt, vocab):

    tokens = clean_text(txt).split()

    token_ids = [vocab.get(token, UNK_ID) for token in tokens]

    return tokens, token_ids

def chunk_sequence(sequence, chunk_size):
    return [
        sequence[i: i + chunk_size]
        for i in range(0, len(sequence), chunk_size)
    ]


In [4]:
cleaned_txt = clean_text(sentence)

tokens = cleaned_txt.split()

vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}

for token in tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

tokens, token_ids = tokenize_encode(sentence, vocab)

chunks = chunk_sequence(token_ids, CHUNK_SIZE)

In [5]:
padded_chunk = pad_sequence(
    [torch.tensor(chunk) for chunk in chunks],
    batch_first = True,
    padding_value = PAD_ID
)

print(padded_chunk)
print(padded_chunk.shape)

tensor([[ 2,  3,  4],
        [ 5,  6,  7],
        [ 8,  9, 10]])
torch.Size([3, 3])


In [6]:
embedding = torch.nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = embedding_dim,
    padding_idx = PAD_ID
)

embedded_chunk = embedding(padded_chunk)

print(embedded_chunk)
print("Shape: ", embedded_chunk.shape)

tensor([[[ 5.4244e-01,  1.1876e+00, -8.5566e-01,  7.7159e-01],
         [ 1.6780e+00,  6.5416e-01,  2.2142e+00, -1.0922e+00],
         [ 1.4707e+00, -3.4531e-01,  3.5581e-01,  7.4944e-01]],

        [[ 8.0224e-02, -1.0828e+00, -5.1863e-01, -8.0181e-01],
         [-8.0177e-01, -5.2905e-01, -1.2592e+00,  6.1168e-01],
         [-5.7300e-01, -1.3922e-01, -1.6288e+00, -1.4823e+00]],

        [[ 1.6746e+00,  3.7607e-01,  3.5413e-01,  2.3437e-01],
         [ 1.6006e+00,  9.4555e-01,  2.5634e-02,  2.9685e-02],
         [ 1.6589e+00,  2.1056e-03,  1.1660e+00,  5.3005e-01]]],
       grad_fn=<EmbeddingBackward0>)
Shape:  torch.Size([3, 3, 4])


In [7]:
mask = padded_chunk.ne(PAD_ID).unsqueeze(-1)

document_vector = (
    (embedded_chunk * mask).sum(dim=(0, 1))
    / mask.sum(dim=(0, 1)).clamp_min(1)
)

print(f"Document vector: {document_vector}")
print(f"Document vector shape: {document_vector.shape}")

Document vector: tensor([ 0.8145,  0.1188, -0.0163, -0.0499], grad_fn=<DivBackward0>)
Document vector shape: torch.Size([4])


In [8]:
num_classes = 2
hidden_layer = 8

classifier = nn.Sequential(
    nn.Linear(embedding_dim, hidden_layer),
    nn.ReLU(),
    nn.Linear(hidden_layer, num_classes)
)

logits = classifier(document_vector)

print("Class scores:", logits)
print("Scores shape:", logits.shape)

Class scores: tensor([-0.1095,  0.0612], grad_fn=<UnsafeViewBackward0>)
Scores shape: torch.Size([2])


In [9]:
documents = [
    "great movie loved it",
    "excellent film very good",
    "terrible movie hated it",
    "awful film very bad",
]

labels = torch.tensor([1, 1, 0, 0], dtype=torch.long)

In [10]:
vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}

for document in documents:
    for token in clean_text(document).split():
        if token not in vocab:
            vocab[token] = len(vocab)

encoded_docs = [
    tokenize_encode(document, vocab)[1]
    for document in documents
]

padded_docs = pad_sequence(
    [torch.tensor(ids) for ids in encoded_docs],
    batch_first=True,
    padding_value = PAD_ID
)

print("Vocab: ", vocab)
print("Padded documents: ", padded_docs)
print("Padded document shape: ", padded_docs.shape)

Vocab:  {'<pad>': 0, '<unk>': 1, 'great': 2, 'movie': 3, 'loved': 4, 'it': 5, 'excellent': 6, 'film': 7, 'very': 8, 'good': 9, 'terrible': 10, 'hated': 11, 'awful': 12, 'bad': 13}
Padded documents:  tensor([[ 2,  3,  4,  5],
        [ 6,  7,  8,  9],
        [10,  3, 11,  5],
        [12,  7,  8, 13]])
Padded document shape:  torch.Size([4, 4])


In [11]:
embedding = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=embedding_dim,
    padding_idx=PAD_ID,
)

In [12]:
embedded_docs = embedding(padded_docs)

print("Embedded chunk: ", embedded_docs)
print("Embedded chunk shape: ", embedded_docs.shape)

Embedded chunk:  tensor([[[ 1.2173,  0.3747,  0.8528, -0.6709],
         [-0.2451,  0.5433, -1.3095,  0.0539],
         [ 1.0464,  0.4211, -1.0970,  0.1937],
         [-0.6380,  0.0898, -0.1309, -0.4647]],

        [[ 0.5785,  2.0905,  1.1540,  1.7890],
         [ 0.2087,  0.0262, -0.0025,  0.1066],
         [ 1.1844,  0.4953, -0.0250,  0.7505],
         [-0.0788,  0.0273, -1.5306,  0.3464]],

        [[-0.1722,  0.5644,  1.9603,  0.8188],
         [-0.2451,  0.5433, -1.3095,  0.0539],
         [ 0.7625, -0.1965, -1.9226, -0.8938],
         [-0.6380,  0.0898, -0.1309, -0.4647]],

        [[ 1.4046, -0.7191, -1.1963,  1.5618],
         [ 0.2087,  0.0262, -0.0025,  0.1066],
         [ 1.1844,  0.4953, -0.0250,  0.7505],
         [-1.7461, -2.4089, -0.9028,  0.2617]]], grad_fn=<EmbeddingBackward0>)
Embedded chunk shape:  torch.Size([4, 4, 4])


In [13]:
mask = padded_docs.ne(PAD_ID).unsqueeze(-1).to(embedded_docs.dtype)

document_vector = (
    (embedded_docs * mask).sum(dim=(1))
    / mask.sum(dim=(0, 1)).clamp_min(1)
)

print("Document vector: ", document_vector)
print("Document vector shape: ", document_vector.shape)

Document vector:  tensor([[ 0.0863,  0.0893, -0.1053, -0.0555],
        [ 0.1183,  0.1650, -0.0253,  0.1870],
        [-0.0183,  0.0626, -0.0877, -0.0304],
        [ 0.0657, -0.1629, -0.1329,  0.1675]], grad_fn=<DivBackward0>)
Document vector shape:  torch.Size([4, 4])


In [14]:
num_classes = 2

#classifier = nn.Linear(embedding_dim, num_classes)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(embedding.parameters()) + list(classifier.parameters()), 
    lr = 0.01
)

logits = classifier(document_vector)

loss = criterion(logits, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Logits shape:", logits.shape)  # (4, 2)
print("Loss:", loss.item())

prediction = logits.argmax(dim=1)

print("Predicted value: ", prediction.tolist())
print("Actual value: ", labels.tolist())
print("Correct:", (prediction == labels).sum().item(), "out of", embedding_dim)

Logits shape: torch.Size([4, 2])
Loss: 0.6870793700218201
Predicted value:  [1, 1, 1, 1]
Actual value:  [1, 1, 0, 0]
Correct: 2 out of 4


In [15]:
for epoch in range(100):
    embedded_docs = embedding(padded_docs)
    mask = padded_docs.ne(PAD_ID).unsqueeze(-1).to(embedded_docs.dtype)
    document_vector = ((embedded_docs * mask).sum(dim=(1))/mask.sum(dim=(0, 1)).clamp_min(1))

    logits = classifier(document_vector)

    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss {loss.item()}")
    

Epoch 20: loss 0.6127853393554688
Epoch 40: loss 0.3708118796348572
Epoch 60: loss 0.09726515412330627
Epoch 80: loss 0.021328317001461983
Epoch 100: loss 0.008737009018659592


In [16]:
embedding.eval()
classifier.eval()

with torch.no_grad():
    embedded_docs = embedding(padded_docs)

    mask = padded_docs.ne(PAD_ID).unsqueeze(-1).to(embedded_docs.dtype)

    document_vector = ((embedded_docs * mask).sum(dim=(1))/ mask.sum(dim=(0, 1)).clamp_min(1))

    logits = classifier(document_vector)

    prediction = logits.argmax(dim=1)

    print("Predicted values: ", prediction.tolist())
    print("Actual_values: ", labels.tolist())
    print("Corrected: ", (prediction == labels).sum().item(), "out of ", embedding_dim)

Predicted values:  [1, 1, 0, 0]
Actual_values:  [1, 1, 0, 0]
Corrected:  4 out of  4
